In [36]:
# libraries
import pandas as pd
import numpy as np
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
from imblearn.under_sampling import RandomUnderSampler
from imblearn.over_sampling import RandomOverSampler, SMOTE
from sklearn.utils.class_weight import compute_class_weight
import warnings
warnings.filterwarnings("ignore")

In [37]:
# load data
df = sns.load_dataset('titanic')
df.head()

,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,NaN,Southampton,no,False
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,C,Cherbourg,yes,False
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,NaN,Southampton,yes,True
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,C,Southampton,yes,False
4,0,3,male,35.0,0,0,8.0500,S,Third,man,True,NaN,Southampton,no,True


In [39]:

df.isnull().sum()

survived         0
pclass           0
sex              0
age            177
sibsp            0
parch            0
fare             0
embarked         2
class            0
who              0
adult_male       0
deck           688
embark_town      2
alive            0
alone            0
dtype: int64

In [40]:
# handle missing values
df.drop(['deck'], axis=1, inplace=True)

median_age = df['age'].median()
df['age'].fillna(median_age, inplace=True)

mode_embarked = df['embarked'].mode()[0]
df['embarked'].fillna(mode_embarked, inplace=True)

columns_to_drop = ['alive', 'who', 'adult_male', 'class', 'embark_town']
df.drop(columns_to_drop, axis=1, inplace=True, errors='ignore')

In [41]:
df.isnull().sum()

survived    0
pclass      0
sex         0
age         0
sibsp       0
parch       0
fare        0
embarked    0
alone       0
dtype: int64

In [42]:
# remove duplicate rows
df = df.drop_duplicates()
df.shape

(775, 9)

In [43]:
print(df["survived"].value_counts())

survived
0    455
1    320
Name: count, dtype: int64


As we can see the class is balanced, so we will make the dataset artificially imbalanced.

In [45]:
# make the data imbalanced
majority = df[df["survived"] == 0]
minority = df[df["survived"] == 1].sample(frac=0.3, random_state=42)

df_imbalanced = pd.concat([majority, minority])

print(df_imbalanced["survived"].value_counts())

survived
0    455
1     96
Name: count, dtype: int64


In [46]:
df_imbalanced.head()

,survived,pclass,sex,age,sibsp,parch,fare,embarked,alone
0,0,3,male,22.0,1,0,7.2500,S,False
4,0,3,male,35.0,0,0,8.0500,S,True
5,0,3,male,28.0,0,0,8.4583,Q,True
6,0,1,male,54.0,0,0,51.8625,S,True
7,0,3,male,2.0,3,1,21.0750,S,False


In [47]:
# preprocess data
X = df_imbalanced.drop("survived", axis=1)
y = df_imbalanced["survived"]

numeric_features = ["pclass", "age", "sibsp", "parch", "fare"]
categorical_features = ["sex", "embarked", "alone"]

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numeric_features),
    ("cat", categorical_pipeline, categorical_features)
])

X_processed = preprocessor.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(
    X_processed,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [54]:
print(pd.Series(y_train).value_counts())

survived
0    363
1     77
Name: count, dtype: int64


Here we first train a base model on our data so that later we can compare the effect of undersampling, oversampling, ...

In [48]:
# baseline model
model = RandomForestClassifier(random_state=42)

model.fit(X_train, y_train)

pred = model.predict(X_test)

print(classification_report(y_test, pred))

              precision    recall  f1-score   support

           0       0.90      0.95      0.92        92
           1       0.64      0.47      0.55        19

    accuracy                           0.86       111
   macro avg       0.77      0.71      0.73       111
weighted avg       0.85      0.86      0.86       111



* Undersampling

In [49]:
# undersampling
rus = RandomUnderSampler(random_state=42)

X_under, y_under = rus.fit_resample(X_train, y_train)

print(pd.Series(y_under).value_counts())

survived
0    77
1    77
Name: count, dtype: int64


Here we applied a random undersampling on our data, so we have 77 samples for both classes.

In [50]:
# undersampling model
under_model = RandomForestClassifier(random_state=42)

under_model.fit(X_under, y_under)

pred_under = under_model.predict(X_test)

print(classification_report(y_test, pred_under))

              precision    recall  f1-score   support

           0       0.91      0.78      0.84        92
           1       0.38      0.63      0.47        19

    accuracy                           0.76       111
   macro avg       0.64      0.71      0.66       111
weighted avg       0.82      0.76      0.78       111



>Random undersampling removes examples from the majority class. As we can see the accuracy has decreased. Because the model has much less information about the majority class, it may misclassify more majority-class examples during testing. As a result, the overall accuracy can decrease, even though the dataset is now balanced. The precision of minority class(1) has also decreased but its recall has increased. The recall improves because the model becomes better at detecting the minority class.

In [51]:
for strategy in [0.5, 0.8, 1.0]:

    rus = RandomUnderSampler(
        sampling_strategy=strategy,
        random_state=42
    )

    X_res, y_res = rus.fit_resample(X_train, y_train)

    model = RandomForestClassifier(random_state=42)

    model.fit(X_res, y_res)

    pred = model.predict(X_test)

    print(f"\nSampling Strategy = {strategy}")
    print(classification_report(y_test, pred))


Sampling Strategy = 0.5
              precision    recall  f1-score   support

           0       0.92      0.87      0.89        92
           1       0.50      0.63      0.56        19

    accuracy                           0.83       111
   macro avg       0.71      0.75      0.73       111
weighted avg       0.85      0.83      0.84       111


Sampling Strategy = 0.8
              precision    recall  f1-score   support

           0       0.92      0.83      0.87        92
           1       0.43      0.63      0.51        19

    accuracy                           0.79       111
   macro avg       0.67      0.73      0.69       111
weighted avg       0.83      0.79      0.81       111


Sampling Strategy = 1.0
              precision    recall  f1-score   support

           0       0.91      0.78      0.84        92
           1       0.38      0.63      0.47        19

    accuracy                           0.76       111
   macro avg       0.64      0.71      0.66       111

>As we can see by increasing the sampling strategy value, the accuracy is decreasing and the recall isn't increasing anymore.

* Oversampling

In [52]:
# oversampling
ros = RandomOverSampler(random_state=42)

X_over, y_over = ros.fit_resample(X_train, y_train)

print(pd.Series(y_over).value_counts())

survived
0    363
1    363
Name: count, dtype: int64


Here we applied a random oversampling on our data, so now we have 363 samples.

In [53]:
# oversampling model
over_model = RandomForestClassifier(random_state=42)

over_model.fit(X_over, y_over)

pred_over = over_model.predict(X_test)

print(classification_report(y_test, pred_over))

              precision    recall  f1-score   support

           0       0.91      0.93      0.92        92
           1       0.62      0.53      0.57        19

    accuracy                           0.86       111
   macro avg       0.77      0.73      0.75       111
weighted avg       0.86      0.86      0.86       111



> Comparing to our base model the recall has increased and the accuracy didn't change. If we compare this with random undersampling, we can see that the accuracy is more in oversampling. Although the recall has increased compared to our imbalanced data, the under sampling method has increased more.
Generally in oversampling method we keep all data and we don't lose information but we may have overfit.

* SMOTE

In [55]:
# smote
smote = SMOTE(random_state=42)

X_smote, y_smote = smote.fit_resample(X_train, y_train)

print(pd.Series(y_smote).value_counts())

survived
0    363
1    363
Name: count, dtype: int64


In [56]:
# smote model
smote_model = RandomForestClassifier(random_state=42)

smote_model.fit(X_smote, y_smote)

pred_smote = smote_model.predict(X_test)

print(classification_report(y_test, pred_smote))

              precision    recall  f1-score   support

           0       0.91      0.93      0.92        92
           1       0.62      0.53      0.57        19

    accuracy                           0.86       111
   macro avg       0.77      0.73      0.75       111
weighted avg       0.86      0.86      0.86       111



## How SMOTE Works

SMOTE (Synthetic Minority Over-sampling Technique)**creates synthetic minority-class samples** instead of simply duplicating existing ones.

### Steps

1. Select a minority-class sample.
2. Find its **k nearest minority neighbors** (default: **k = 5**).
3. Randomly choose one of these neighbors.
4. Generate a new sample between the selected sample and its neighbor.

This process creates realistic synthetic observations that help the classifier learn a broader and more generalized decision boundary.

Smote vs. Random Oversampling

| **Random Oversampling**              | **SMOTE**                              |
| ------------------------------------ | -------------------------------------- |
| Duplicates existing minority samples | Creates new synthetic minority samples |
| Higher risk of overfitting           | Lower risk of overfitting              |
| Simple and fast to implement         | More computationally intensive         |
| Does not add new information         | Adds diversity to the minority class   |


* Cost-Sensitive Learning

In [57]:
# compute weights
weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(y_train),
    y=y_train
)

class_weights = {
    0: weights[0],
    1: weights[1]
}

print(class_weights)

{0: np.float64(0.6060606060606061), 1: np.float64(2.857142857142857)}


In [58]:
# weighted model
weighted_model = RandomForestClassifier(class_weight=class_weights, random_state=42)

weighted_model.fit(X_train, y_train)

pred_weighted = weighted_model.predict(X_test)

print(classification_report(y_test, pred_weighted))

              precision    recall  f1-score   support

           0       0.90      0.96      0.93        92
           1       0.69      0.47      0.56        19

    accuracy                           0.87       111
   macro avg       0.80      0.72      0.74       111
weighted avg       0.86      0.87      0.86       111



>Comparing to our base model the accuracy has increased and recall has also increased but not as much as oversampling and undersampling methods.

## Comparing performances

Random Undersampling balances the classes by removing majority-class samples, which can improve minority detection but may reduce overall accuracy due to information loss.

Random Oversampling balances the data by duplicating minority-class samples. It preserves all original data but can increase the risk of overfitting.

SMOTE generates synthetic minority-class samples using nearest neighbors, often providing a better balance between minority recall and generalization than simple oversampling.

Cost-Sensitive Learning adjusts the classifier to penalize mistakes on the minority class more heavily, improving minority-class performance without modifying the dataset itself. It is often a strong baseline because it retains all training data while addressing class imbalance.